In [1]:
import numpy as np
import pandas as pd
import os
import glob


In [2]:
import warnings

warnings.filterwarnings(
    "ignore",
    message=".*select_dtypes.*object.*deprecated.*",
    category=FutureWarning,
)

In [3]:
folder_path = os.getcwd()
dataset_folder_path = os.path.join(folder_path, "..\\data_raw\\")


### Files & Folders to be processed
* CPI(Consumer Price Index) for understanding inflation faced by consumers & purchasing powers
* Events file consists of events that happened from 2008 - 2026
* Interest rates file for analyzing change in interest rates over time due to events.
* Market_index folder which contains Nifty Midcap files
* News file covers news headlines that impacted Indian stock market spanning from 2008 - 2026
* Sector Indices folder which contains various sectors data is there in folder
* WPI(Wholesale price index) for understanding producer inflation

### First we will analyze market & sector indices data

In [4]:
# Function for profiling data of given file
def profiling_data(file):
    # Reading file
    if file.endswith(".csv"):
        df = pd.read_csv(file)

    elif file.endswith(".xlsx"):
        df = pd.read_excel(file)

    elif file.endswith(".parquet"):
        df = pd.read_parquet(file)

    # Step 1: Basic dataset information
    print("=================Basic dataset information========================")
    # Shape
    print("Shape:", df.shape)

    # Column names
    print("Columns list:\n", df.columns)

    # Data types
    print(df.dtypes)

    # Info summary
    print(df.info())

    # Step 2: Missing Value Analysis
    print("\n======================Missing Value Analysis========================")
    # Missing values count
    missing_count = df.isnull().sum()

    # Missing percentage
    missing_percent = (df.isnull().sum() / len(df)) * 100

    missing_df = pd.DataFrame({
        "Missing Count": missing_count,
        "Missing %": missing_percent
    })

    print(missing_df.sort_values("Missing %", ascending=False))

    # Step 3: Duplicate Records
    print("\n========================Duplicate Records=============================")
    duplicates = df.duplicated().sum()
    print("Duplicate rows:", duplicates)
    print("Duplicate %:", duplicates / len(df) * 100)

    # Step 4. Numerical Column Statistics
    print("\n======================Numerical Column Statistics=================================")
    numeric_summary = df.describe().T
    numeric_summary["skewness"] = df.skew(numeric_only=True)
    numeric_summary["kurtosis"] = df.kurtosis(numeric_only=True)
    print(numeric_summary)

    # Step:5 Categorical Column Analysis
    print("\n===============================Categorical Column Analysis============================")
    cat_cols = df.select_dtypes(include="object").columns

    for col in cat_cols:
        print(f"\nColumn: {col}")
        print("Unique values:", df[col].nunique())
        # print(df[col].value_counts().head(10))

    # Step:6 Outlier Detection
    print("\n==================Outlier Detection=============================")
    # Using IQR:
    num_cols = df.select_dtypes(include=['int64', 'float64']).columns

    for col in num_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        outliers = df[(df[col] < lower) | (df[col] > upper)]
        print(f"{col}: {len(outliers)} outliers")

    # Step:7 Data Type Consistency
    # What to profile
    # Numeric stored as string
    # Date columns
    # Mixed types

    print("\n====================Data Type Consistency=======================")
    for col in df.columns:
        print(f"{df[col].apply(type).value_counts()}\n")


### Profiling market index data

In [5]:
import glob
# Profiling market data
market_data_folder = os.path.join(dataset_folder_path, "market_index")
files_list = glob.glob(os.path.join(market_data_folder, "*.csv"))
i = 1

for file in files_list:
    print(
        f"=========================================Analysing FILE:{i}====================================================")
    profiling_data(file)
    print("===================END OF PROFILING==================================")
    print()
    i += 1


=========================================Analysing FILE:1====================================================
=================Basic dataset information========================
Shape: (246, 5)
Columns list:
 Index(['Date', 'Open', 'High', 'Low', 'Close'], dtype='object')
Date      object
Open      object
High      object
Low       object
Close    float64
dtype: object
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 246 entries, 0 to 245
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Date    246 non-null    object 
 1   Open    246 non-null    object 
 2   High    246 non-null    object 
 3   Low     246 non-null    object 
 4   Close   246 non-null    float64
dtypes: float64(1), object(4)
memory usage: 9.7+ KB
None

======================Missing Value Analysis========================
       Missing Count  Missing %
Date               0        0.0
Open               0        0.0
High               0        0.0
Low        

### Observations
* Here some files contain missing values for dates as pandas is not able to parse
* When the OHLC value is supposed to be float still its being taken as string these values are required to be fixed

## Profiling sector indices data

In [6]:
sector_data_folder = os.path.join(dataset_folder_path, "sector_indices")
print(os.listdir(sector_data_folder))


['nifty_auto', 'nifty_bank', 'nifty_energy', 'nifty_financial_services', 'nifty_fmcg', 'nifty_it', 'nifty_metal', 'nifty_pharma', 'nifty_realty']


* Here as we can see there are indices folder inside main folder
* We need to analyze each folder iteratively inside the main folder

In [7]:
sectors_list = os.listdir(sector_data_folder)
print(len(sectors_list))


9


Since we have 9 folders to analyze we will split each profiling in bin size of 3

In [8]:
# Analysing first 3 folders
folder_num = 1

for i in range(0, 3):
    sector_folder = os.path.join(sector_data_folder, sectors_list[i])
    files_list = glob.glob(os.path.join(sector_folder, "*.csv"))
    file_num = 1

    for file in files_list:
        print(
            f"========================Analysing FOLDER:{folder_num} FILE:{file_num}================================")
        profiling_data(file)

        print()
        file_num += 1

    print(
        f"===================END OF PROFILING FOLDER {folder_num}==================================")
    folder_num += 1


========================Analysing FOLDER:1 FILE:1================================
=================Basic dataset information========================
Shape: (3688, 5)
Columns list:
 Index(['Date', 'Open', 'High', 'Low', 'Close'], dtype='object')
Date     object
Open     object
High     object
Low      object
Close    object
dtype: object
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3688 entries, 0 to 3687
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Date    3688 non-null   object
 1   Open    3688 non-null   object
 2   High    3688 non-null   object
 3   Low     3688 non-null   object
 4   Close   3688 non-null   object
dtypes: object(5)
memory usage: 144.2+ KB
None

======================Missing Value Analysis========================
       Missing Count  Missing %
Date               0        0.0
Open               0        0.0
High               0        0.0
Low                0        0.0
Close              0      

### Analyzing next 3 sectors data

In [9]:
folder_num = 4

# As python calculates index from 0
for i in range(3, 6):
    sector_folder = os.path.join(sector_data_folder, sectors_list[i])
    files_list = glob.glob(os.path.join(sector_folder, "*.csv"))
    file_num = 1

    for file in files_list:
        print(
            f"========================Analysing FOLDER:{folder_num} FILE:{file_num}================================")
        profiling_data(file)

        print()
        file_num += 1

    print(
        f"===================END OF PROFILING FOLDER {folder_num}==================================")
    folder_num += 1


========================Analysing FOLDER:4 FILE:1================================
=================Basic dataset information========================
Shape: (4073, 5)
Columns list:
 Index(['Date', 'Open', 'High', 'Low', 'Close'], dtype='object')
Date     object
Open     object
High     object
Low      object
Close    object
dtype: object
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4073 entries, 0 to 4072
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Date    4073 non-null   object
 1   Open    4073 non-null   object
 2   High    4073 non-null   object
 3   Low     4073 non-null   object
 4   Close   4073 non-null   object
dtypes: object(5)
memory usage: 159.2+ KB
None

======================Missing Value Analysis========================
       Missing Count  Missing %
Date               0        0.0
Open               0        0.0
High               0        0.0
Low                0        0.0
Close              0      

In [10]:
folder_num = 7

for i in range(6, 9):
    sector_folder = os.path.join(sector_data_folder, sectors_list[i])
    files_list = glob.glob(os.path.join(sector_folder, "*.csv"))
    file_num = 1

    for file in files_list:
        print(
            f"========================Analysing FOLDER:{folder_num} FILE:{file_num}================================")
        profiling_data(file)

        print()
        file_num += 1

    print(
        f"===================END OF PROFILING FOLDER {folder_num}====================================================")
    print()
    folder_num += 1


========================Analysing FOLDER:7 FILE:1================================
=================Basic dataset information========================
Shape: (3690, 7)
Columns list:
 Index(['Date', 'Close', 'Open', 'High', 'Low', 'Vol.', 'Change %'], dtype='object')
Date        object
Close       object
Open        object
High        object
Low         object
Vol.        object
Change %    object
dtype: object
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3690 entries, 0 to 3689
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Date      3690 non-null   object
 1   Close     3690 non-null   object
 2   Open      3690 non-null   object
 3   High      3690 non-null   object
 4   Low       3690 non-null   object
 5   Vol.      2700 non-null   object
 6   Change %  3690 non-null   object
dtypes: object(7)
memory usage: 201.9+ KB
None

======================Missing Value Analysis========================
          Missing Count

### Observations
* Here also some files contain missing values for dates as pandas is not able to parse
* When the OHLC value is supposed to be float still its being taken as string these values are required to be fixed

### Analyzing below individual files
* Interest rates
* CPI
* WPI
* news 
* Events

In [11]:
os.listdir(dataset_folder_path)


['CPI.xlsx',
 'events.xlsx',
 'interest_rates.xlsx',
 'market_index',
 'news.xlsx',
 'sector_indices',
 'WPI.xlsx']

In [12]:
# Since all this files are in xlsx here we will use .xlsx
files_list = glob.glob(os.path.join(dataset_folder_path, "*.xlsx"))
for file in files_list:
    profiling_data(file)
    print("=========================================== End of PROFILING================================================")
    print()


=================Basic dataset information========================
Shape: (170, 4)
Columns list:
 Index(['Release Date', 'Actual', 'Forecast', 'Previous'], dtype='object')
Release Date     object
Actual          float64
Forecast        float64
Previous        float64
dtype: object
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 170 entries, 0 to 169
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Release Date  170 non-null    object 
 1   Actual        170 non-null    float64
 2   Forecast      169 non-null    float64
 3   Previous      170 non-null    float64
dtypes: float64(3), object(1)
memory usage: 5.4+ KB
None

======================Missing Value Analysis========================
              Missing Count  Missing %
Forecast                  1   0.588235
Release Date              0   0.000000
Actual                    0   0.000000
Previous                  0   0.000000

========================Duplicate

### Observations
* Here for files like events don't have headers
* Here date columns are of type object which needs to be fixed
* Also since the types are not getting detected many times columns are reflecting as null